# 3ptWL-mod emulator likelihood for Firecrown

This notebook shows how to wrap the trained 3ptWL-mod emulator as a Gaussian likelihood. For now the covariance is an artificial diagonal matrix; later you can replace it with the covariance estimated from simulations, keeping the same data-vector ordering.

The flow is:

1. Load the local 3ptWL-mod emulator trained by `tests/emulator.ipynb`.
2. Select the generated test-set vector closest to a Planck-like cosmology and apply the new binning mask.
3. Build a Gaussian likelihood with an invented covariance.
4. Run a compact `emcee` fit as a sanity check.
5. Show the Firecrown-style `Statistic` and `ConstGaussian` scaffold.

Firecrown normally uses factory functions and SACC data. Here we keep the 3ptWL-mod emulator likelihood explicit so it is easy to verify before connecting the real covariance and sampler configuration.

In [ ]:
from __future__ import annotations

import importlib.util
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.linalg import cho_factor, cho_solve
from scipy.optimize import least_squares

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 160,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

In [ ]:
def find_repo_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for path in [start, *start.parents]:
        if (path / "source").exists() and (path / "tests" / "emulator.py").exists():
            return path
    raise RuntimeError("Could not find the wlcf-documented repository root.")


REPO_ROOT = find_repo_root()
TEST_DIR = REPO_ROOT / "tests"

spec = importlib.util.spec_from_file_location("wlcf_documented_emulator", TEST_DIR / "emulator.py")
flow = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = flow
spec.loader.exec_module(flow)

config = flow.EmulatorConfig()
paths = flow.make_paths(config, repo_root=REPO_ROOT)

print(f"Repository: {REPO_ROOT}")
print(f"Weights path: {paths.weights_path}")
print(f"Vector directory: {paths.vector_dir}")

In [ ]:
def module_available(name: str) -> bool:
    return importlib.util.find_spec(name) is not None


FIRECROWN_AVAILABLE = module_available("firecrown")
SACC_AVAILABLE = module_available("sacc")
EMCEE_AVAILABLE = module_available("emcee")
CORNER_AVAILABLE = module_available("corner")

print("Optional packages")
print(f"  firecrown: {FIRECROWN_AVAILABLE}")
print(f"  sacc     : {SACC_AVAILABLE}")
print(f"  emcee    : {EMCEE_AVAILABLE}")
print(f"  corner   : {CORNER_AVAILABLE}")

if not FIRECROWN_AVAILABLE:
    print("Install Firecrown later with something like: python -m pip install firecrown sacc")

## Load the emulator

Run `tests/emulator.ipynb` first if the weights file does not exist. The emulator returns all trained multipoles on the interpolated new binning.

In [ ]:
if not paths.weights_path.exists():
    raise FileNotFoundError(
        f"Missing emulator weights: {paths.weights_path}\n"
        "Run tests/emulator.ipynb first, then rerun this notebook."
    )

emulator = flow.WLCFEmulator(paths.weights_path)

bounds_df = pd.DataFrame(
    emulator.bounds,
    index=emulator.param_names,
    columns=["lower", "upper"],
)

print("Parameters:", emulator.param_names)
print("Moments:", emulator.moments)
print("Target shape:", emulator.target_shape)
print("Theta bins:", emulator.n_theta)
display(bounds_df)

## Select a Planck-Like Mock Data Vector

For this prototype the data vector is the generated test-set 3ptWL-mod vector closest to a Planck-like reference cosmology. The likelihood then compares emulator predictions against that vector. If the generated vectors are missing, the notebook falls back to an emulator prediction at the Planck-like reference point so the rest of the notebook still demonstrates the interface.

In [ ]:
rng = np.random.default_rng(20260609)
planck_like_params = flow.PLANCK_2018_REFERENCE
planck_theta = emulator.theta_array(planck_like_params)

try:
    grid, X_test, sample_id, truth, target_path = flow.choose_test_sample(
        config,
        paths,
        emulator,
        rng=rng,
        target_params=planck_like_params,
    )
    observed_full = flow.load_target_vector(target_path, emulator)
    data_source = f"generated Planck-like test sample {sample_id:04d}"
except Exception as exc:
    warnings.warn(f"Could not load a generated test vector ({exc}). Falling back to emulator Planck-like mock data.")
    sample_id = -1
    truth = planck_theta.copy()
    observed_full = emulator.predict_vector(truth)
    data_source = "emulator prediction at the Planck-like reference point"

theory_full_at_truth = emulator.predict_vector(truth)
truth_table = pd.DataFrame(
    {
        "Planck-like reference": planck_theta,
        "selected truth": truth,
        "delta": truth - planck_theta,
    },
    index=emulator.param_names,
)
normalized_delta = (truth - planck_theta) / (emulator.bounds[:, 1] - emulator.bounds[:, 0])

print(f"Data source: {data_source}")
print(f"Full vector length: {observed_full.size}")
print(f"Normalized distance to reference: {np.linalg.norm(normalized_delta):.4f}")
display(truth_table)

## Apply the new binning mask

The mask selects a compact set of matrix entries from each multipole. This is the ordering expected by the artificial covariance below:

`moment 0 selected entries`, then `moment 1 selected entries`, and so on.

In [ ]:
if emulator.target_representation != "full_matrix":
    selected_indices = np.arange(observed_full.size, dtype=int)
    mask_2d = None
    print("The emulator target is not a full matrix representation; using every vector entry.")
else:
    mask_2d = flow.new_binning_mask(dim=emulator.n_theta, symm=True)
    entries_per_moment = emulator.n_theta * emulator.n_theta
    selected_indices = []
    for moment_index, moment in enumerate(emulator.moments):
        start = moment_index * entries_per_moment
        selected_indices.extend(start + np.flatnonzero(mask_2d.ravel()))
    selected_indices = np.asarray(selected_indices, dtype=int)

observed_vector = observed_full[selected_indices]
theory_vector_at_truth = theory_full_at_truth[selected_indices]

print(f"Selected entries per moment: {len(selected_indices) // len(emulator.moments)}")
print(f"Likelihood data-vector length: {observed_vector.size}")

if mask_2d is not None:
    fig_mask, _ = flow.plot_new_binning_mask()
    plt.show()

## Invent an artificial covariance

Replace this cell with the real covariance when it is ready. The only requirement is that the covariance uses the same selected-vector ordering defined above.

For this demo the fractional error is intentionally broad so the triangle plot shows the true value inside the posterior contours. Do not interpret this covariance as a forecast constraint.

In [ ]:
FRACTIONAL_ERROR = 1.30  # Broad demo covariance; replace with the real covariance later.
ERROR_FLOOR = 1.0e-13

sigma = FRACTIONAL_ERROR * np.maximum(np.abs(observed_vector), ERROR_FLOOR)
covariance = np.diag(sigma**2)

print(f"Covariance shape: {covariance.shape}")
print(f"Smallest sigma: {sigma.min():.3e}")
print(f"Largest sigma : {sigma.max():.3e}")

## Gaussian likelihood core

This class is the numerical object we want Firecrown to call. It takes parameters, evaluates the 3ptWL-mod emulator, applies the same selected indices, and computes the Gaussian log-likelihood using a covariance matrix.

In [ ]:
class WLCFEmulatorGaussianLikelihood:
    def __init__(self, emulator, selected_indices, data_vector, covariance):
        self.emulator = emulator
        self.selected_indices = np.asarray(selected_indices, dtype=int)
        self.data_vector = np.asarray(data_vector, dtype=float)
        self.covariance = np.asarray(covariance, dtype=float)
        self.param_names = list(emulator.param_names)
        self.lower = emulator.bounds[:, 0]
        self.upper = emulator.bounds[:, 1]

        if self.covariance.shape != (self.data_vector.size, self.data_vector.size):
            raise ValueError("Covariance shape does not match data-vector length.")

        self._cov_factor = cho_factor(self.covariance, lower=True, check_finite=False)
        chol = self._cov_factor[0]
        self.logdet = 2.0 * np.sum(np.log(np.diag(chol)))
        self.norm = self.data_vector.size * np.log(2.0 * np.pi) + self.logdet

    def theta_array(self, params):
        return self.emulator.theta_array(params)

    def in_bounds(self, theta) -> bool:
        theta = self.theta_array(theta)
        return bool(np.all(theta >= self.lower) and np.all(theta <= self.upper))

    def theory(self, params) -> np.ndarray:
        full_vector = self.emulator.predict_vector(params)
        return full_vector[self.selected_indices]

    def residual(self, params) -> np.ndarray:
        return self.data_vector - self.theory(params)

    def chi2(self, params) -> float:
        residual = self.residual(params)
        weighted = cho_solve(self._cov_factor, residual, check_finite=False)
        return float(residual @ weighted)

    def log_prior(self, params) -> float:
        return 0.0 if self.in_bounds(params) else -np.inf

    def loglike(self, params, include_normalization: bool = False) -> float:
        lp = self.log_prior(params)
        if not np.isfinite(lp):
            return -np.inf
        value = -0.5 * self.chi2(params)
        if include_normalization:
            value += -0.5 * self.norm
        return float(value)

    def log_probability(self, params) -> float:
        lp = self.log_prior(params)
        if not np.isfinite(lp):
            return -np.inf
        return lp + self.loglike(params)


likelihood = WLCFEmulatorGaussianLikelihood(
    emulator=emulator,
    selected_indices=selected_indices,
    data_vector=observed_vector,
    covariance=covariance,
)

print(f"chi2 at truth: {likelihood.chi2(truth):.3f}")
print(f"loglike at truth: {likelihood.loglike(truth):.3f}")

In [ ]:
residual_in_sigma = (theory_vector_at_truth - observed_vector) / sigma

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)

ax = axes[0]
ax.plot(observed_vector, color="0.20", lw=1.1, label="mock data")
ax.plot(theory_vector_at_truth, color="#1f77b4", lw=1.0, alpha=0.85, label="emulator at true parameters")
ax.set_xlabel("selected data-vector index")
ax.set_ylabel(r"$\zeta_m$")
ax.set_title("Selected 3ptWL-mod vector")
ax.legend(frameon=False)

ax = axes[1]
ax.hist(residual_in_sigma, bins=30, color="0.25", edgecolor="white", lw=0.6)
ax.axvline(0.0, color="#1f77b4", lw=1.5)
ax.set_xlabel(r"$(\mathrm{emulator} - \mathrm{data}) / \sigma$")
ax.set_ylabel("count")
ax.set_title("Residuals under artificial covariance")

plt.show()

## Fit the mock vector with `emcee`

This is a notebook-level sanity check. In a full Firecrown workflow, the sampler would call the Firecrown likelihood object instead.

In [ ]:
RUN_MCMC = True
N_WALKERS = 24
N_STEPS = 220
BURN_IN = 70

if RUN_MCMC and not EMCEE_AVAILABLE:
    raise ImportError("Install emcee first: python -m pip install emcee")

if RUN_MCMC:
    import emcee

    lower = likelihood.lower
    upper = likelihood.upper
    ndim = len(emulator.param_names)

    def whitened_residual(theta):
        return (likelihood.theory(theta) - likelihood.data_vector) / sigma

    start = np.clip(truth + rng.normal(0.0, 0.02 * (upper - lower)), lower, upper)
    lsq = least_squares(whitened_residual, start, bounds=(lower, upper), max_nfev=400)
    best_theta = lsq.x

    proposal_width = np.maximum(0.003 * (upper - lower), 1.0e-4)
    initial = best_theta + rng.normal(0.0, proposal_width, size=(N_WALKERS, ndim))
    initial = np.clip(initial, lower + 1.0e-9, upper - 1.0e-9)

    sampler = emcee.EnsembleSampler(N_WALKERS, ndim, likelihood.log_probability)
    sampler.run_mcmc(initial, N_STEPS, progress=True)
    chain = sampler.get_chain(discard=BURN_IN, flat=True)

    print("Mean acceptance fraction:", np.mean(sampler.acceptance_fraction))
    print("Best fit:")
    display(pd.Series(best_theta, index=emulator.param_names, name="least_squares_best").to_frame())
else:
    chain = None
    best_theta = None
    print("MCMC skipped.")

In [ ]:
if chain is not None:
    rows = []
    for i, name in enumerate(emulator.param_names):
        q16, q50, q84 = np.percentile(chain[:, i], [16, 50, 84])
        rows.append({
            "parameter": name,
            "truth": truth[i],
            "median": q50,
            "minus_1sigma": q50 - q16,
            "plus_1sigma": q84 - q50,
            "median_minus_truth": q50 - truth[i],
        })
    summary = pd.DataFrame(rows)
    display(summary)

In [ ]:
if chain is not None:
    if not CORNER_AVAILABLE:
        raise ImportError("Install corner first: python -m pip install corner")

    import corner

    corner_fig = corner.corner(
        chain,
        labels=emulator.param_names,
        show_titles=True,
        title_fmt=".5f",
        quantiles=[0.16, 0.5, 0.84],
    )

    truth_color = "#1f77b4"
    axes = np.asarray(corner_fig.axes).reshape(len(emulator.param_names), len(emulator.param_names))
    for row in range(len(emulator.param_names)):
        axes[row, row].axvline(truth[row], color=truth_color, lw=2.0, zorder=20)
        for col in range(row):
            axes[row, col].axvline(truth[col], color=truth_color, lw=1.6, zorder=20)
            axes[row, col].axhline(truth[row], color=truth_color, lw=1.6, zorder=20)
            axes[row, col].plot(truth[col], truth[row], marker="o", ms=4, color=truth_color, zorder=25)

    corner_fig.text(0.62, 0.92, "blue = true value", color=truth_color, fontsize=12)
    plt.show()

## Firecrown scaffold

Firecrown's Gaussian likelihood family uses `Statistic` objects and a `ConstGaussian` likelihood. The factory below wraps the numerical likelihood core above. It is guarded because Firecrown is optional in this notebook environment.

For a production run, move this class and `build_likelihood` function into a small Python module, replace the artificial covariance, and connect the sampled parameters through the sampler interface you choose.

In [ ]:
if FIRECROWN_AVAILABLE:
    try:
        from firecrown.likelihood import Statistic, ConstGaussian
    except ImportError:
        from firecrown.likelihood import Statistic
        from firecrown.likelihood.gaussian import ConstGaussian

    class WLCFEmulatorStatistic(Statistic):
        # Firecrown statistic wrapper around the 3ptWL-mod emulator likelihood core.
        def __init__(self, core_likelihood, parameter_prefix=None):
            super().__init__(parameter_prefix=parameter_prefix)
            self.core_likelihood = core_likelihood
            self.sacc_indices = np.arange(core_likelihood.data_vector.size)
            self.ready = True

        def read(self, sacc_data=None):
            # The prototype already receives data and covariance directly.
            # A production SACC version would read indices/data here.
            super().read(sacc_data)

        def get_data_vector(self):
            return self.core_likelihood.data_vector

        def compute_theory_vector(self, tools):
            # This small adapter accepts either a dict-like object or an object with
            # attributes named Omega_m, h, and logAs. A sampler connector can fill
            # these values before Firecrown evaluates the likelihood.
            try:
                theta = np.array([tools[name] for name in self.core_likelihood.param_names], dtype=float)
            except (TypeError, KeyError):
                theta = np.array([getattr(tools, name) for name in self.core_likelihood.param_names], dtype=float)
            self.theory_vector = self.core_likelihood.theory(theta)
            self.computed_theory_vector = True
            return self.theory_vector

    def build_likelihood(build_parameters=None):
        statistic = WLCFEmulatorStatistic(likelihood)
        firecrown_likelihood = ConstGaussian.create_ready([statistic], covariance)
        return firecrown_likelihood, None

    firecrown_likelihood, firecrown_tools = build_likelihood()
    print("Built Firecrown ConstGaussian likelihood:", firecrown_likelihood)
else:
    print("Firecrown is not installed. The likelihood core above is ready; install Firecrown to activate this scaffold.")

## Replacing the artificial covariance

When the real covariance is available, replace the covariance cell with something like:

```python
covariance = np.load("/path/to/covariance.npy")
likelihood = WLCFEmulatorGaussianLikelihood(
    emulator=emulator,
    selected_indices=selected_indices,
    data_vector=observed_vector,
    covariance=covariance,
)
```

The critical detail is ordering: the covariance rows and columns must follow the same `selected_indices` order used here. If the covariance was built from simulation vectors, apply the same new binning mask to every simulation vector before estimating the covariance.